In [1]:
import keras.layers
import unicodedata
import numpy as np
import pandas as pd
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import Conv2D,Dense,Dropout,Input,LSTM,Embedding,MultiHeadAttention,LayerNormalization
import os
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
import datasets
from datasets import Dataset,DatasetDict
import tensorflow as tf
import re
from tensorflow.keras.preprocessing.sequence import pad_sequences
import ctypes

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

In [2]:
ctypes.windll.kernel32.SetThreadExecutionState(0x80000002)

In [3]:
with open ("D:/project_deeplearning/TEP.en-fa.en",encoding="utf-8") as f:
    en_s=f.read().splitlines()

with open ("D:/project_deeplearning/TEP.en-fa.fa",encoding="utf-8") as f:
    fa_s=f.read().splitlines()


assert len(en_s)==len(fa_s)

df=pd.DataFrame(
    {
        "en":en_s,
        "fa":fa_s
    }
)

df=df.sample(n=550000,random_state=42)
df=df.reset_index(drop=True)

en=df["en"]
fa=df["fa"]
data=pd.DataFrame({"train":Dataset.from_pandas(df)})
print(data["train"][0])

In [4]:
train=data["train"]
train[:10]

In [5]:
def unicode_to_asci(s):
    return "".join(c for c in unicodedata.normalize("NFC",s) if unicodedata.category(c)!='Mn' )

In [6]:
len(data)

In [7]:
def preprossing(w):
    w=unicode_to_asci(w.lower().strip())
    w = re.sub(r"([.!?])", r" \1 ", w)
    w=re.sub(r'([""])+',"",w)
    w=w.rstrip().strip()
    w = "<start> " + w + " <end>"
    return w

In [8]:
en_sen="Im very happy."
preprossing(en_sen)

In [9]:
fa_sen="درود بر تو."
preprossing(fa_sen)

In [10]:
df["en"]=df["en"].apply(preprossing)
df["fa"]=df["fa"].apply(preprossing)

In [11]:
max_length=35
batch_size=128

trainer=BpeTrainer(vocab_size=5000)

token_en=Tokenizer(BPE(unk_token="<unk>"))
token_fa=Tokenizer(BPE(unk_token="<unk>"))

token_en.add_special_tokens(["<pad>", "<unk>", "<start>", "<end>"])
token_fa.add_special_tokens(["<pad>", "<unk>", "<start>", "<end>"])
token_en.pre_tokenizer=Whitespace()
token_fa.pre_tokenizer=Whitespace()

token_en.train_from_iterator(df["en"].tolist(), trainer)
token_fa.train_from_iterator(df["fa"].tolist(), trainer)

def encode(tokenizer, text):
    return tokenizer.encode(text).ids


In [12]:
en_seq=[encode(token_en, t) for t in df["en"]]
fa_seq=[encode(token_fa, t) for t in df["fa"]]

en_seq=pad_sequences(en_seq,maxlen=max_length,padding="post",truncating="post")
fa_seq=pad_sequences(fa_seq,maxlen=max_length,padding="post",truncating="post")

inputs=fa_seq[:, :-1]
targets=fa_seq[:, 1:]

decoder_seq_lenght=max_length-1
vocab_size_en=token_en.get_vocab_size()
vocab_size_fa=token_fa.get_vocab_size()

In [29]:
num_layer=4
d_model=256
num_he=8
d_ff=512
dropout_rate=0.1

def positional_encoder(seq_len,d_model):
    positional=np.arange(seq_len)[:,np.newaxis]
    dims=np.arange(d_model)[np.newaxis,:]
    angle_rate=1/np.power(10000,(2*(dims//2))/np.float32(d_model))
    angl=positional*angle_rate
    angl[:,0::2]=np.sin(angl[:,0::2])
    angl[:,1::2]=np.cos(angl[:,1::2])
    return tf.cast(angl[np.newaxis,...],tf.float32)



In [30]:
def feed_forward_n(d_model,d_ff):
    return tf.keras.Sequential([
        Dense(d_ff,activation='relu'),
        Dense(d_model)
    ])

In [42]:
class encoderlayer(tf.keras.layers.Layer):
    def __init__(self,d_model,num_he,d_ff,dropout_rate):
        super().__init__()
        self.mha=MultiHeadAttention(num_heads=num_he,key_dim=d_model//num_he)
        self.ffn=feed_forward_n(d_model,d_ff)
        self.norm1=LayerNormalization(epsilon=0.000001)
        self.norm2=LayerNormalization(epsilon=0.000001)
        self.dropout1=Dropout(dropout_rate)
        self.dropout2=Dropout(dropout_rate)

    def call(self,x,msk,training):
        attn_out=self.mha(query=x,key=x,value=x,attention_mask=msk)
        attn_out=self.dropout1(attn_out,training=training)
        out1=self.norm1(x+attn_out)

        ffn_outp=self.ffn(out1)
        ffn_outp=self.dropout2(ffn_outp,training=training)
        outp2=self.norm2(out1+ffn_outp)

        return outp2

In [43]:
class decoderlayer(tf.keras.layers.Layer):
    def __init__(self,d_model,num_he,d_ff,dropout_rate):
        super().__init__()
        self.mha1=MultiHeadAttention(num_heads=num_he,key_dim=d_model//num_he)
        self.mha2=MultiHeadAttention(num_heads=num_he,key_dim=d_model//num_he)

        self.ffn=feed_forward_n(d_model, d_ff)
        self.norm1=LayerNormalization(epsilon=0.000001)
        self.norm2=LayerNormalization(epsilon=0.000001)
        self.norm3=LayerNormalization(epsilon=0.000001)
        self.dropout1=Dropout(dropout_rate)
        self.dropout2=Dropout(dropout_rate)
        self.dropout3=Dropout(dropout_rate)

        def call(self,x,msk,training,look_ahead_msk,padding_mask):
            attn1=self.mha1(query=x,key=x,value=x,attention_mask=msk)
            attn1=self.dropout1(attn1,training=training)
            out1=self.norm1(attn1+x)

            attn2=self.mha2(query=out1,key=out1,value=out1,attention_mask=msk)
            attn2=self.dropout2(attn1,training=training)
            out2=self.norm2(attn2+out1)

            ffn_output=self.ffn(out2)
            ffn_output=self.dropout3(ffn_output,training=training)
            out3=self.norm3(ffn_output+out2)


            return out3


In [16]:
model.summary()

In [17]:
n=len(en_seq)
train_end=int(n*0.85)
val_end=int(n * 0.95)

en_train=en_seq[:train_end]
en_val=en_seq[train_end:val_end]
en_test=en_seq[val_end:]

fa_train=fa_seq[:train_end]
fa_val=fa_seq[train_end:val_end]
fa_test=fa_seq[val_end:]

inputs_train=fa_train[:, :-1]
targets_train=fa_train[:, 1:]

inputs_val=fa_val[:, :-1]
targets_val=fa_val[:, 1:]

inputs_test=fa_test[:, :-1]
targets_test=fa_test[:, 1:]

train_ds=tf.data.Dataset.from_tensor_slices(((en_train, inputs_train), targets_train)).shuffle(30000).batch(
    batch_size).prefetch(tf.data.AUTOTUNE)
val_ds=tf.data.Dataset.from_tensor_slices(((en_val, inputs_val), targets_val)).batch(batch_size).prefetch(
    tf.data.AUTOTUNE)
test_ds=tf.data.Dataset.from_tensor_slices(((en_test, inputs_test), targets_test)).batch(batch_size).prefetch(
    tf.data.AUTOTUNE)

In [18]:
loss_ob=tf.keras.losses.SparseCategoricalCrossentropy(reduction="none",from_logits=False)

In [19]:
def msk_loss(y_true,y_pred):

    loss=loss_ob(y_true,y_pred)

    mask=tf.cast(tf.not_equal(y_true,0),dtype=loss.dtype)
    loss=loss*mask
    return tf.reduce_sum(loss)/tf.reduce_sum(mask)

In [20]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),metrics=[],loss=msk_loss)

In [21]:
early=EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)


In [22]:
print(f"number of english sentences: {len(en_s)}")
print(f"number of persian sentences: {len(fa_s)}")
print(f"are they equal? {len(en_s)==len(fa_s)}")

In [23]:
import random
sample_idx=random.sample(range(len(en_s)), 15)
for i in sample_idx:
    print(f"EN: {en_s[i]}")
    print(f"FA: {fa_s[i]}")
    print("---")

In [24]:
en_lengths = [len(encode(token_en, t)) for t in df["en"].tolist()]
fa_lengths = [len(encode(token_fa, t)) for t in df["fa"].tolist()]

en_over = sum(1 for l in en_lengths if l > max_length)
fa_over = sum(1 for l in fa_lengths if l > max_length)

print(f"Current max_length: {max_length}")
print(f"English sentences longer than max_length: {en_over} ({en_over / len(en_lengths) * 100:.1f}%)")
print(f"Persian sentences longer than max_length: {fa_over} ({fa_over / len(fa_lengths) * 100:.1f}%)")
print(
    f"English mean length: {np.mean(en_lengths):.1f} | 95th percentile: {np.percentile(en_lengths, 95):.1f} | max: {np.max(en_lengths)}")
print(
    f"Persian mean length: {np.mean(fa_lengths):.1f} | 95th percentile: {np.percentile(fa_lengths, 95):.1f} | max: {np.max(fa_lengths)}")


long_idx_en = [i for i, l in enumerate(en_lengths) if l > max_length][:3]

for i in long_idx_en:
    original_ids = encode(token_en, df["en"].iloc[i])
    padded = pad_sequences([original_ids], maxlen=max_length, padding="post")[0]

    print(f"original sentence: {df['en'].iloc[i]}")
    print(f"actual token count: {len(original_ids)}")
    print(f"original tokens first 10: {[token_en.id_to_token(t) for t in original_ids[:10]]}")
    print(f"after pad_sequences with max_length={max_length}: {[token_en.id_to_token(t) for t in padded if t != 0]}")
    print(f"is <start> still present? {token_en.token_to_id('<start>') in padded}")
    print("---")

In [25]:

long_idx_fa = [i for i, l in enumerate(fa_lengths) if l > max_length][:3]

for i in long_idx_fa:
    original_ids = encode(token_fa, df["fa"].iloc[i])
    padded = pad_sequences([original_ids], maxlen=max_length, padding="post")[0]

    print(f"original sentence: {df['fa'].iloc[i]}")
    print(f"actual token count: {len(original_ids)}")
    print(f"original tokens first 10: {[token_fa.id_to_token(t) for t in original_ids[:10]]}")
    print(f"after pad_sequences with max_length={max_length}: {[token_fa.id_to_token(t) for t in padded if t != 0]}")
    print(f"is <start> still present? {token_fa.token_to_id('<start>') in padded}")
    print("---")

In [26]:
l=ReduceLROnPlateau(monitor="val_loss",patience=2,min_lr=0.000001,verbose=1,factor=0.5)

In [27]:
history=model.fit(train_ds,epochs=120,validation_data=val_ds,verbose=2,callbacks=[early,l])

In [30]:
assert len(en_s)==len(fa_s)

In [26]:
print(token_fa.encode("<start> سلام <end>").tokens)

In [24]:

reverse_fa = {idx: token_fa.id_to_token(idx) for idx in range(token_fa.get_vocab_size())}

In [25]:
import pickle
from tensorflow.keras.models import load_model

model=load_model("Translator.keras",custom_objects={"msk_loss":msk_loss})

In [26]:
import pickle
model=load_model("Translator.keras")
pickle.dump(token_fa,open("token_fa.pkl","wb"))
pickle.dump(token_en,open("token_en.pkl","wb"))
pickle.dump(reverse_fa,open("reverse_fa.pkl","wb"))

In [27]:
import pickle

token_fa=pickle.load(open("token_fa.pkl","rb"))
token_en=pickle.load(open("token_en.pkl","rb"))
reverse_fa=pickle.load(open("reverse_fa.pkl","rb"))


In [35]:
encoder_model=tf.keras.Model(encoder_inputs,[encoder_output,state_h,state_c])

In [ ]:
decoder_inputs_inf=Input(shape=(1,),name="decoder_inputs_inf")
decoder_emb2=decoder_embedding(decoder_inputs_inf)

decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    decoder_emb2,
    initial_state=decoder_states_inputs
)
attn2=attention_layers(query=decoder_outputs2,key=enc_out_input,value=enc_out_input)
x2=decoder_outputs2+attn2
x2=nor(x2)
decoder_outputs2=decoder_dense(x2)

decoder_model=tf.keras.Model(
    [decoder_inputs_inf,decoder_state_input_h,decoder_state_input_c,enc_out_input],
    [decoder_outputs2, state_h2, state_c2]
)

In [37]:
encoder_model.save('encoder_model.keras')
decoder_model.save('decoder_model.keras')

In [43]:
def translate(sentence):
    sentence = preprossing(sentence)

    seq=[token_en.encode(sentence).ids]
    seq=pad_sequences(seq, maxlen=max_length, padding="post")

    enc_out, h, c=encoder_model.predict(seq)

    enc_out=tf.convert_to_tensor(enc_out)
    h=tf.convert_to_tensor(h)
    c=tf.convert_to_tensor(c)

    start_token_id=token_fa.token_to_id("<start>")

    target_seq=tf.constant([[start_token_id]])

    stop=False
    decoded = ""

    while not stop:
        output_tokens, h, c =decoder_model([target_seq, h, c, enc_out])

        sampled_token_index=np.argmax(output_tokens[0, -1, :])
        sampled_word=reverse_fa.get(sampled_token_index, "")

        if sampled_word=="<end>" or sampled_word == "" or len(decoded.split())>max_length:
            stop = True
        else:
            decoded += " " +sampled_word

        target_seq=tf.constant([[sampled_token_index]])

    return decoded.strip()

In [44]:
print(translate("I love you"))


In [45]:
print(translate('you can'))

In [46]:
import sacrebleu
txt="من دوست دارم"
reverse=[" دوستت دارم ","من تورو دوست دارم"]
blu=sacrebleu.sentence_bleu(txt,reverse,tokenize="none")
print(f"score:{blu.score:.2f}")

In [ ]:
print(tf.config.list_physical_devices('GPU'))

In [49]:
import sacrebleu
import re
import random

test_en=df["en"].tolist()[-test_size:]
test_fa=df["fa"].tolist()[-test_size:]

def clean_text(text):
    text=text.replace("<start>", "").replace("<end>", "")
    text=re.sub(r"([.!?،؟])", r" \1", text)
    return re.sub(r"\s+", " ", text).strip()

model_translations=[]
for i, en_sent in enumerate(test_en[:200]):
    translated=translate(en_sent)
    model_translations.append(clean_text(translated))
    if (i+1)%50==0:
        print(f"{i+1}")

references=[[clean_text(ref)] for ref in test_fa[:200]]

bleu=sacrebleu.corpus_bleu(
    model_translations,
    references,
    tokenize='none'
)

print("\n" + "="*50)
print(f"BLEU Score:{bleu.score:.2f}")
print(f"Brevity Penalty:{bleu.bp:.3f}")
print(f"Length Ratio:{bleu.sys_len/bleu.ref_len:.3f}")
print("="*50)